In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle


In [24]:
# load datasets
data=pd.read_csv("Churn_Modelling.csv")
data.head

<bound method NDFrame.head of       RowNumber  CustomerId    Surname  CreditScore Geography  Gender  Age  \
0             1    15634602   Hargrave          619    France  Female   42   
1             2    15647311       Hill          608     Spain  Female   41   
2             3    15619304       Onio          502    France  Female   42   
3             4    15701354       Boni          699    France  Female   39   
4             5    15737888   Mitchell          850     Spain  Female   43   
...         ...         ...        ...          ...       ...     ...  ...   
9995       9996    15606229   Obijiaku          771    France    Male   39   
9996       9997    15569892  Johnstone          516    France    Male   35   
9997       9998    15584532        Liu          709    France  Female   36   
9998       9999    15682355  Sabbatini          772   Germany    Male   42   
9999      10000    15628319     Walker          792    France  Female   28   

      Tenure    Balance  NumOfPro

In [25]:
# preprocess the data
# drop irrelevant columns

data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data



,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [26]:
# ab dekho ki agr koi aisa data h jisko encode krne kijaruruat h 
# like categorical way me male female,yes no aisa type
# kyuki model ko number smjh ata h names nhi..isisliye 
# baaki number columns h wo apen aap smjh lega

label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])
data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [27]:
# here on geography column we will apply ohe we cant do
# labelencoder as it is a having other the 2 value,so 
# model will make unnecessary heirarchy

from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
ohe_geo=ohe.fit_transform(data[['Geography']])
ohe_geo.toarray()


array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [28]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [97]:
# yaha maine features name ke column banaye aur usme vectorized georgrpahy daldiya
geo_df=pd.DataFrame(ohe_geo.toarray(),columns=ohe.get_feature_names_out(['Geography']))

In [32]:
data=pd.concat([data.drop('Geography',axis=1),geo_df],axis=1)

In [30]:
# now we will save the robot which are labelEncoder and OneHotEncoder
# into pickle file

with open ('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('ohe.pkl','wb') as file:
    pickle.dump(ohe,file)

In [ ]:
# now we will separate the dependet and independent features

x=data.drop('Exited',axis=1)
y=data['Exited']

# split the data in training and testing sets
# x_train + y_train → model.fit() → model seekhta hai
# x_test → model.predict() → guess karta hai
# y_test → actual answer
# compare → accuracy

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# scale these features mtlb same unit me lana taki moel confuse na ho ki mtr bada h cm se for eg

scaler=StandardScaler()
x_train = scaler.fit_transform(x_train)  # learn + apply kyuki x traijn se hi na trianing krna h to fit transform taki wo learn or apply kre..
# x test me sirf apply krnah kyuki uss etest krna h usko learn nhi krwana h soch ke dekh...iska ans dega tbhina pata chlega shi bana h
x_test = scaler.transform(x_test)

In [35]:
x_test

array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.81311987, -1.09499335,  0.77030065, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.41876609,  0.91324755, -0.94100321, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.24540869,  0.91324755,  0.00972116, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [36]:
with open('scaler.pkl',"wb") as file:
    pickle.dump(scaler,file)

In [37]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [39]:
x_train.shape[1]

12

In [40]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [41]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    # hidden layer 1 connected to input layer
    Dense(32,activation='relu'),
    # hl2
    Dense(1,activation='sigmoid')
    # output layer
])

In [42]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [43]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()

In [44]:
# compiling the model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [ ]:
# set up the tensorboard for log visual thorugh grpah
log_dir="logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [56]:
# setting up earlystopping 
early_stopping_callbacks=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [62]:
# troin the model
model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,callbacks=[tensorflow_callback]
)


Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 0.2048 - accuracy: 0.9025 - val_loss: 0.7132 - val_accuracy: 0.8480
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.2009 - accuracy: 0.9007 - val_loss: 0.7448 - val_accuracy: 0.8475
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.1978 - accuracy: 0.9034 - val_loss: 0.7001 - val_accuracy: 0.8450
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.1941 - accuracy: 0.9049 - val_loss: 0.7501 - val_accuracy: 0.8455
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.1899 - accuracy: 0.9060 - val_loss: 0.7374 - val_accuracy: 0.8440
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.1991 - accuracy: 0.9018 - val_loss: 0.7693 - val_accuracy: 0.8400
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.1963 - accuracy: 0.9022 - val_loss: 0.7765 - val_accuracy: 0.8465

In [63]:
model.save('model.h5')

c:\UDEMY\ANN CLASSIFICATION\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [68]:
# load tensor board extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [69]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 25976), started 0:17:37 ago. (Use '!kill 25976' to kill it.)

In [70]:
from tensorflow.keras.models import load_model
import numpy as np


In [ ]:
# now here we willl load all the trained model,pickle file of ohe gender and label enocder geo

model=load_model('model.h5')

with open('ohe.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)


with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)    

with open('scaler.pkl','rb') as file:
    scaler_load=pickle.load(file)

In [72]:
# now we will take the sample input

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [ ]:
# now we will encode geo here first as it it required
geo_encoded=label_encoder_geo.transform([[input_data['Geography']]]).toarray( )
geo_encoded_df=pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\UDEMY\ANN CLASSIFICATION\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [86]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [87]:
# now we will encode gender
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [88]:
# now we will concat the geo with the input df

input_df=pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
# s aling the input now
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [92]:
# prediction of churn

prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 26ms/step


array([[1.0258722e-15]], dtype=float32)

In [94]:
prediction_prob=prediction[0][0]
prediction_prob

1.0258722e-15

In [95]:
if prediction_prob > 0.5:
   print('customer is likely to churn')
else:
   print('cusotomer is not likely to churn')

cusotomer is not likely to churn


In [ ]:
# DATA LOAD → CLEAN → ENCODE (Gender + Geography)
# → SAVE ENCODERS
# → SPLIT (TRAIN/TEST)
# → SCALE DATA
# → BUILD MODEL (ANN)
# → COMPILE
# → TRAIN (fit)
# → SAVE MODEL

# PREDICTION SIDE:
# LOAD MODEL + ENCODERS + SCALER
# → RAW INPUT
# → ENCODE
# → SCALE
# → PREDICT
# → RESULT